# Power Analysis - Acquifer Data (7d)

Post-hoc power analysis for the Acquifer spheroid dataset (Size and GFP measurements) at the 7d timepoint, using `power_analysis_functions.R`. Two comparisons per measurement:
1. **Laser vs. Control**, at each timepoint (before/after/7d)
2. **Timepoint vs. timepoint**, within each treatment

A Bonferroni correction is applied across all comparisons for a given measurement. Produces one Excel workbook per measurement (Size, GFP) with a Results and a Parameters sheet.

## Step 0: Libraries

In [ ]:
# Install required packages (run once)
# install.packages("pwr", repos="https://cloud.r-project.org/")
# install.packages("readxl", repos="https://cloud.r-project.org/")
# install.packages("writexl", repos="https://cloud.r-project.org/")
# install.packages("lsr", repos="https://cloud.r-project.org/")

library(pwr)
library(readxl)
library(writexl)
library(lsr)
library(dplyr)
library(stringr)

## Step 1: Source shared functions

In [ ]:
base_dir <- getwd()
source(file.path(base_dir, "Functions", "power_analysis_functions.R"))

## Step 2: Load raw data and set analysis parameters

Bonferroni correction is computed once here (from the Size dataset) and reused for the GFP analysis below, on the assumption both datasets have the same number of timepoints/treatments and therefore the same comparison count.

In [ ]:
data_path <- file.path(base_dir, "Rawdata_Files", "Datatable_Summary_7d.xlsx")
df_size <- read_excel(data_path, sheet = "Size")
df_GFP  <- read_excel(data_path, sheet = "GFP")

# Analysis parameters
desired_power <- 0.8
alpha_level <- 0.05

timepoints <- unique(df_size$timepoint)
treatments <- unique(df_size$treatment)

# Multiple testing correction (Bonferroni), applied across both comparisons:
#   Comparison 1: Laser vs Control, once per timepoint
#   Comparison 2: 3 timepoint pairs, once per treatment
n_comparisons_1 <- length(timepoints)
n_comparisons_2 <- 3 * length(treatments)
n_total_comparisons <- n_comparisons_1 + n_comparisons_2
alpha_corrected <- alpha_level / n_total_comparisons

cat("Timepoints:", paste(timepoints, collapse=", "), "\n")
cat("Treatments:", paste(treatments, collapse=", "), "\n")
cat("\n--- Multiple Testing Correction ---\n")
cat("Total number of comparisons:", n_total_comparisons, "\n")
cat("  - Comparison 1 (Treatment):", n_comparisons_1, "tests\n")
cat("  - Comparison 2 (Timepoint):", n_comparisons_2, "tests\n")
cat("Original alpha:", alpha_level, "\n")
cat("Bonferroni-corrected alpha:", round(alpha_corrected, 5), "\n")

# Timepoint pairs compared within each treatment for Comparison 2
timepoint_pairs <- list(
  c("before", "after"),
  c("before", "7d"),
  c("after", "7d")
)

## Size dataset: Comparison 1 - Laser vs Control

In [ ]:
cat("\n", strrep("=", 70), "\n")
cat("Size - COMPARISON 1: LASER vs CONTROL (by timepoint)\n")
cat(strrep("=", 70), "\n\n")
cat("Using Bonferroni-corrected alpha:", round(alpha_corrected, 5), "\n\n")

results_size_treatment <- perform_power_analysis(
  data = df_size,
  measurement_col = "Size",
  treatment_col = "treatment",
  group1 = "Control",
  group2 = "Laser",
  grouping_vars = c("timepoint"),
  power = desired_power,
  alpha = alpha_corrected,
  analysis_name = "Size: Laser vs Control by Timepoint"
)

print(results_size_treatment)

## Size dataset: Comparison 2 - Timepoint vs Timepoint

In [ ]:
cat("\n", strrep("=", 70), "\n")
cat("Size - COMPARISON 2: TIMEPOINT vs TIMEPOINT (within each treatment)\n")
cat(strrep("=", 70), "\n\n")

results_size_timepoint <- run_timepoint_comparisons(
  data = df_size,
  measurement_col = "Size",
  group_levels = treatments,
  timepoint_pairs = timepoint_pairs,
  power = desired_power,
  alpha = alpha_corrected,
  label_prefix = "Size: "
)

cat("\n", strrep("=", 70), "\n")
cat("SUMMARY: Size Timepoint Comparisons\n")
cat(strrep("=", 70), "\n\n")
print(results_size_timepoint)

## Size dataset: Save results

In [ ]:
output_path_size <- file.path(base_dir, "Report_Files", "power_analysis_Size_results_7d.xlsx")

save_power_results(
  treatment_result = results_size_treatment,
  treatment_group1 = "Control",
  treatment_group2 = "Laser",
  timepoint_result = results_size_timepoint,
  output_path = output_path_size,
  measurement_col = "Size",
  data_source = paste0("Rawdata_Files/Datatable_Summary_7d.xlsx (Sheet: Size)"),
  desired_power = desired_power,
  alpha_level = alpha_level,
  n_total_comparisons = n_total_comparisons,
  alpha_corrected = alpha_corrected,
  comparison1_desc = paste0("Laser vs Control (for each timepoint: before, after, 7d) - ", n_comparisons_1, " tests"),
  comparison2_desc = paste0("Timepoint vs Timepoint within each treatment (before vs after, before vs 7d, after vs 7d) - ", n_comparisons_2, " tests")
)

## GFP dataset: Comparison 1 - Laser vs Control

In [ ]:
cat("\n", strrep("=", 70), "\n")
cat("GFP - COMPARISON 1: LASER vs CONTROL (by timepoint)\n")
cat(strrep("=", 70), "\n\n")
cat("Using Bonferroni-corrected alpha:", round(alpha_corrected, 5), "\n\n")

results_gfp_treatment <- perform_power_analysis(
  data = df_GFP,
  measurement_col = "GFP",
  treatment_col = "treatment",
  group1 = "Control",
  group2 = "Laser",
  grouping_vars = c("timepoint"),
  power = desired_power,
  alpha = alpha_corrected,
  analysis_name = "GFP: Laser vs Control by Timepoint"
)

print(results_gfp_treatment)

## GFP dataset: Comparison 2 - Timepoint vs Timepoint

In [ ]:
cat("\n", strrep("=", 70), "\n")
cat("GFP - COMPARISON 2: TIMEPOINT vs TIMEPOINT (within each treatment)\n")
cat(strrep("=", 70), "\n\n")

results_gfp_timepoint <- run_timepoint_comparisons(
  data = df_GFP,
  measurement_col = "GFP",
  group_levels = treatments,
  timepoint_pairs = timepoint_pairs,
  power = desired_power,
  alpha = alpha_corrected,
  label_prefix = "GFP: "
)

cat("\n", strrep("=", 70), "\n")
cat("SUMMARY: GFP Timepoint Comparisons\n")
cat(strrep("=", 70), "\n\n")
print(results_gfp_timepoint)

## GFP dataset: Save results

In [ ]:
output_path_gfp <- file.path(base_dir, "Report_Files", "power_analysis_GFP_results_7d.xlsx")

save_power_results(
  treatment_result = results_gfp_treatment,
  treatment_group1 = "Control",
  treatment_group2 = "Laser",
  timepoint_result = results_gfp_timepoint,
  output_path = output_path_gfp,
  measurement_col = "GFP",
  data_source = paste0("Rawdata_Files/Datatable_Summary_7d.xlsx (Sheet: GFP)"),
  desired_power = desired_power,
  alpha_level = alpha_level,
  n_total_comparisons = n_total_comparisons,
  alpha_corrected = alpha_corrected,
  comparison1_desc = paste0("Laser vs Control (for each timepoint: before, after, 7d) - ", n_comparisons_1, " tests"),
  comparison2_desc = paste0("Timepoint vs Timepoint within each treatment (before vs after, before vs 7d, after vs 7d) - ", n_comparisons_2, " tests")
)